In [40]:
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")

import deepchem as dc

In [62]:
import deepchem as dc

print("DeepChem version:", dc.__version__)

DeepChem version: 2.8.0


In [42]:
tasks, datasets, transformers = dc.molnet.load_delaney()

train_dataset, valid_dataset, test_dataset = datasets

print("预测任务：", tasks)
print("训练集样本数：", len(train_dataset))
print("验证集样本数：", len(valid_dataset))
print("测试集样本数：", len(test_dataset))

预测任务： ['measured log solubility in mols per litre']
训练集样本数： 902
验证集样本数： 113
测试集样本数： 113


In [63]:
print("训练集 X 的形状：", train_dataset.X.shape)
print("训练集 y 的形状：", train_dataset.y.shape)
print("训练集 ids 的形状：", train_dataset.ids.shape)

print("\n前 5 个 SMILES：")
for smiles in train_dataset.ids[:5]:
    print(smiles)

print("\n前 10 个溶解度标签：")
print(train_dataset.y[:10].flatten())

训练集 X 的形状： (902, 1024)
训练集 y 的形状： (902, 1)
训练集 ids 的形状： (902,)

前 5 个 SMILES：
CC(C)=CCCC(C)=CC(=O)
CCCC=C
CCCCCCCCCCCCCC
CC(C)Cl
CCC(C)CO

前 10 个溶解度标签：
[ 0.39041294  0.09042128 -2.46434643  0.70492033  1.15974639 -0.03538168
 -0.60149498  0.51137732 -0.14666891 -0.91358308]


In [44]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [45]:
X_train = train_dataset.X
y_train = train_dataset.y.flatten()

X_valid = valid_dataset.X
y_valid = valid_dataset.y.flatten()

X_test = test_dataset.X
y_test = test_dataset.y.flatten()

print(X_train.shape, y_train.shape)
print(X_valid.shape, y_valid.shape)
print(X_test.shape, y_test.shape)

(902, 1024) (902,)
(113, 1024) (113,)
(113, 1024) (113,)


In [46]:
tree_model = DecisionTreeRegressor(
    random_state=42
)

tree_model.fit(X_train, y_train)

print("决策树训练完成")

决策树训练完成


In [47]:
valid_pred = tree_model.predict(X_valid)

mae = mean_absolute_error(y_valid, valid_pred)
rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
r2 = r2_score(y_valid, valid_pred)

print("验证集 MAE：", mae)
print("验证集 RMSE：", rmse)
print("验证集 R²：", r2)

验证集 MAE： 0.7897004317004468
验证集 RMSE： 1.0320724210084158
验证集 R²： -0.1642442204884338


In [48]:
train_pred = tree_model.predict(X_train)

train_mae = mean_absolute_error(y_train, train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
train_r2 = r2_score(y_train, train_pred)

print("训练集 MAE：", train_mae)
print("训练集 RMSE：", train_rmse)
print("训练集 R²：", train_r2)

训练集 MAE： 0.025048375499019362
训练集 RMSE： 0.14168688529785567
训练集 R²： 0.9799248265345922


In [49]:
tree_limited = DecisionTreeRegressor(
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

tree_limited.fit(X_train, y_train)

valid_pred_limited = tree_limited.predict(X_valid)

mae_limited = mean_absolute_error(y_valid, valid_pred_limited)
rmse_limited = np.sqrt(
    mean_squared_error(y_valid, valid_pred_limited)
)
r2_limited = r2_score(y_valid, valid_pred_limited)

print("限制后验证集 MAE：", mae_limited)
print("限制后验证集 RMSE：", rmse_limited)
print("限制后验证集 R²：", r2_limited)

限制后验证集 MAE： 0.7825821496037059
限制后验证集 RMSE： 1.0212067955752893
限制后验证集 R²： -0.13985901129797362


In [50]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("随机森林训练完成")

随机森林训练完成


In [51]:
rf_valid_pred = rf_model.predict(X_valid)

rf_mae = mean_absolute_error(y_valid, rf_valid_pred)
rf_rmse = np.sqrt(mean_squared_error(y_valid, rf_valid_pred))
rf_r2 = r2_score(y_valid, rf_valid_pred)

print("随机森林验证集 MAE：", rf_mae)
print("随机森林验证集 RMSE：", rf_rmse)
print("随机森林验证集 R²：", rf_r2)

随机森林验证集 MAE： 0.6351758815070396
随机森林验证集 RMSE： 0.8244335220037031
随机森林验证集 R²： 0.2570920103793888


In [52]:
rf_train_pred = rf_model.predict(X_train)

rf_train_mae = mean_absolute_error(y_train, rf_train_pred)
rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rf_train_r2 = r2_score(y_train, rf_train_pred)

print("随机森林训练集 MAE：", rf_train_mae)
print("随机森林训练集 RMSE：", rf_train_rmse)
print("随机森林训练集 R²：", rf_train_r2)

随机森林训练集 MAE： 0.3725384811045116
随机森林训练集 RMSE： 0.49258276933764844
随机森林训练集 R²： 0.7573622153516528


In [53]:
rf_test_pred = rf_model.predict(X_test)

rf_test_mae = mean_absolute_error(y_test, rf_test_pred)
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))
rf_test_r2 = r2_score(y_test, rf_test_pred)

print("随机森林测试集 MAE：", rf_test_mae)
print("随机森林测试集 RMSE：", rf_test_rmse)
print("随机森林测试集 R²：", rf_test_r2)

随机森林测试集 MAE： 0.6449177374558237
随机森林测试集 RMSE： 0.8356065730547443
随机森林测试集 R²： 0.33618523342298756


In [54]:
import pandas as pd

results = pd.DataFrame({
    "模型": [
        "原始决策树",
        "限制后决策树",
        "随机森林"
    ],
    "验证集_MAE": [
        mae,
        mae_limited,
        rf_mae
    ],
    "验证集_RMSE": [
        rmse,
        rmse_limited,
        rf_rmse
    ],
    "验证集_R2": [
        r2,
        r2_limited,
        rf_r2
    ]
})

results

,模型,验证集_MAE,验证集_RMSE,验证集_R2
0,原始决策树,0.789700,1.032072,-0.164244
1,限制后决策树,0.782582,1.021207,-0.139859
2,随机森林,0.635176,0.824434,0.257092


In [55]:
results.to_csv(
    "model_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("结果已保存")

结果已保存
